## **Ecoregion Map of South America**
-----
#### SDS 210 - Programming with Spatial Data

*Author: Isabelle Bartholet*

*Date: May 2026*

### **Research Question**
* Are fires linked to Ecoregions? Do certain ecoregions have more fires than others?
* Is there a correlation between fires and ecoregions in South America?


### **Content**

1. Load important packages 
2. Call map key via function check_map_key()
3. Fetch the fire data via function fetch_data()
4. Convert Fire Data to gdf
5. Import Ecoregions Shapefile
6. Do some Analysis I guess

5. Data handling (Rescaling and Filtering)
6. Create Map with manipulated FRP Data and bright_ti4 as colour

------
#### **Important Libraries and Map Key** 

In [2]:
import requests
import pandas as pd
import geopandas as gpd
import time
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import folium
import contextily
import cmcrameri
import branca.colormap as cm
import numpy as np



from cartopy import crs as ccrs
from geodatasets import get_path

from config import MAP_KEY
from access_mapkey import check_map_key

#### **Access Map Key**

In [3]:
check_map_key(MAP_KEY)

transaction_limit             5000
current_transactions             0
transaction_interval    10 minutes
dtype: object


{'transaction_limit': 5000,
 'current_transactions': 0,
 'transaction_interval': '10 minutes'}

#### **Import Data for the Last Five Days**

In [4]:
from FetchFireData2 import fetch_data
from config import MAP_KEY

df_fire = fetch_data(MAP_KEY)

# check length of df_fire, to see whether it worked correctly
print(f"There are {len(df_fire)} rows in this data frame.")

# quickly check dataframe by using head
df_fire.head(6)


Request successful


 Data download successful! 7177 fires found.
There are 7177 rows in this data frame.


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
0,-5.44360,-36.79174,303.93,0.41,0.61,2026-05-14,313,N20,VIIRS,n,2.0NRT,285.17,1.14,N
1,-12.40172,-38.34732,302.24,0.40,0.60,2026-05-14,315,N20,VIIRS,n,2.0NRT,288.83,1.39,N
2,-12.39697,-38.34255,309.97,0.40,0.60,2026-05-14,315,N20,VIIRS,n,2.0NRT,289.85,1.18,N
3,-12.39624,-38.34624,305.83,0.40,0.60,2026-05-14,315,N20,VIIRS,n,2.0NRT,289.12,1.18,N
4,-9.46632,-40.34198,347.44,0.61,0.71,2026-05-14,315,N20,VIIRS,n,2.0NRT,288.26,31.16,N
5,-9.46524,-40.34753,315.09,0.61,0.71,2026-05-14,315,N20,VIIRS,n,2.0NRT,287.40,43.93,N


#### **Convert Fire Dataframe into GDF and Check for NAs**

In [5]:
# create gdf 
gdf_fire = gpd.GeoDataFrame(
    df_fire, 
    geometry=gpd.points_from_xy(
        df_fire.longitude, df_fire.latitude),
        
        crs="EPSG:4326")

# check, to see if gdf returns the same information as the df above 
gdf_fire.head(6)


# check for NAs
gdf_fire.info()
print(f"There are no NAs in this gdf.") ## noch ändern zu etwas, was mit gdf_fire.info() zusammenhängt


<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 7177 entries, 0 to 7176
Data columns (total 15 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   latitude    7177 non-null   float64 
 1   longitude   7177 non-null   float64 
 2   bright_ti4  7177 non-null   float64 
 3   scan        7177 non-null   float64 
 4   track       7177 non-null   float64 
 5   acq_date    7177 non-null   str     
 6   acq_time    7177 non-null   int64   
 7   satellite   7177 non-null   str     
 8   instrument  7177 non-null   str     
 9   confidence  7177 non-null   str     
 10  version     7177 non-null   str     
 11  bright_ti5  7177 non-null   float64 
 12  frp         7177 non-null   float64 
 13  daynight    7177 non-null   str     
 14  geometry    7177 non-null   geometry
dtypes: float64(7), geometry(1), int64(1), str(6)
memory usage: 1023.4 KB
There are no NAs in this gdf.


#### **Load Worldwide Shapefile**

1. Load worldwide Shapefile -> a lot of data
2. Filter only polygons for South America 



In [6]:
## load worldwide shapefile
gdf_world_ecoregions = gpd.read_file("../data/Ecoregions2017.zip")

## filter for only polygons in on South America; column REALM == "Neotropic"
gdf_ecoregions_sa = gdf_world_ecoregions[(gdf_world_ecoregions["REALM"]=="Neotropic")].copy()

gdf_ecoregions_sa.head(6)


,OBJECTID,ECO_NAME,BIOME_NUM,BIOME_NAME,REALM,ECO_BIOME_,NNH,ECO_ID,SHAPE_LENG,SHAPE_AREA,NNH_NAME,COLOR,COLOR_BIO,COLOR_NNH,LICENSE,geometry
21,22.0,Alto Paraná Atlantic forests,1.0,Tropical & Subtropical Moist Broadleaf Forests,Neotropic,NO01,4,439,205.740939,42.742563,Nature Imperiled,#267400,#38A700,#EE1E23,CC-BY 4.0,"MULTIPOLYGON (((-52.31779 -28.42464, -52.29635..."
22,23.0,Amazon-Orinoco-Southern Caribbean mangroves,14.0,Mangroves,Neotropic,NO14,1,611,139.824908,3.346216,Half Protected,#E600AA,#FE01C4,#257339,CC-BY 4.0,"MULTIPOLYGON (((-44.58923 -3.0014, -44.59199 -..."
36,37.0,Apure-Villavicencio dry forests,2.0,Tropical & Subtropical Dry Broadleaf Forests,Neotropic,NO02,3,520,44.540782,5.587565,Nature Could Recover,#ABE038,#CCCD65,#F9A91B,CC-BY 4.0,"MULTIPOLYGON (((-74.26596 2.95891, -74.25538 2..."
38,39.0,Araucaria moist forests,1.0,Tropical & Subtropical Moist Broadleaf Forests,Neotropic,NO01,3,440,68.503449,19.521012,Nature Could Recover,#70A800,#38A700,#F9A91B,CC-BY 4.0,"MULTIPOLYGON (((-52.22039 -29.32238, -52.23367..."
39,40.0,Araya and Paria xeric scrub,13.0,Deserts & Xeric Shrublands,Neotropic,NO13,4,597,10.258896,0.434596,Nature Imperiled,#B29841,#CC6767,#EE1E23,CC-BY 4.0,"MULTIPOLYGON (((-62.74015 10.7487, -62.75399 1..."
46,47.0,Atacama desert,13.0,Deserts & Xeric Shrublands,Neotropic,NO13,2,598,18.786639,9.200860,Nature Could Reach Half Protected,#F18650,#CC6767,#7BC141,CC-BY 4.0,"POLYGON ((-69.42981 -25.18036, -69.49561 -25.3..."


In [7]:
## check crs and geommetry
print(gdf_ecoregions_sa.geometry.head())
print(gdf_ecoregions_sa.crs)  

21    MULTIPOLYGON (((-52.31779 -28.42464, -52.29635...
22    MULTIPOLYGON (((-44.58923 -3.0014, -44.59199 -...
36    MULTIPOLYGON (((-74.26596 2.95891, -74.25538 2...
38    MULTIPOLYGON (((-52.22039 -29.32238, -52.23367...
39    MULTIPOLYGON (((-62.74015 10.7487, -62.75399 1...
Name: geometry, dtype: geometry
EPSG:4326


#### **Simplify the Polygons**

Simplify the polygons, so the map is easier to load and to work with.

Source for simplification code: https://scikit-geometry.github.io/scikit-geometry/simplify.html

In [8]:
gdf_ecoregions_sa["geometry"] = gdf_ecoregions_sa.geometry.simplify(
    tolerance = 0.05,
    preserve_topology = True
).copy()

gdf_ecoregions_sa.head(4)


,OBJECTID,ECO_NAME,BIOME_NUM,BIOME_NAME,REALM,ECO_BIOME_,NNH,ECO_ID,SHAPE_LENG,SHAPE_AREA,NNH_NAME,COLOR,COLOR_BIO,COLOR_NNH,LICENSE,geometry
21,22.0,Alto Paraná Atlantic forests,1.0,Tropical & Subtropical Moist Broadleaf Forests,Neotropic,NO01,4,439,205.740939,42.742563,Nature Imperiled,#267400,#38A700,#EE1E23,CC-BY 4.0,"MULTIPOLYGON (((-52.31779 -28.42464, -52.19617..."
22,23.0,Amazon-Orinoco-Southern Caribbean mangroves,14.0,Mangroves,Neotropic,NO14,1,611,139.824908,3.346216,Half Protected,#E600AA,#FE01C4,#257339,CC-BY 4.0,"MULTIPOLYGON (((-44.58923 -3.0014, -44.65545 -..."
36,37.0,Apure-Villavicencio dry forests,2.0,Tropical & Subtropical Dry Broadleaf Forests,Neotropic,NO02,3,520,44.540782,5.587565,Nature Could Recover,#ABE038,#CCCD65,#F9A91B,CC-BY 4.0,"MULTIPOLYGON (((-74.35937 3.05428, -74.2162 2...."
38,39.0,Araucaria moist forests,1.0,Tropical & Subtropical Moist Broadleaf Forests,Neotropic,NO01,3,440,68.503449,19.521012,Nature Could Recover,#70A800,#38A700,#F9A91B,CC-BY 4.0,"MULTIPOLYGON (((-52.22039 -29.32238, -52.27636..."


#### **Create a Simple Ecoregion's Map**

In [9]:

# initialize basemap centerd on South America, add a zoom control limit
min_lon, max_lon = -81.5, -35.0
min_lat, max_lat = -65.0, 14.5


eco_region_base = folium.Map(
    max_bounds = True,
    location = [15, -69],  #7 = latitute, 7 degrees north of equator, -65 = longitude, 65 degrees west of the prime meridian
    zoom_start = 6, 
    tiles = "CartoDB Positron",
    control_scale = True,
    min_lat = min_lat,
    max_lat = max_lat,
    min_lon = min_lon,
    max_lon = max_lon
                         
)

## add choropleth map
folium.Choropleth(
    geo_data=gdf_ecoregions_sa,
    data = gdf_ecoregions_sa,
    name = "Biome Types South America",
    columns=['ECO_ID', 'BIOME_NUM'],
    key_on='feature.properties.ECO_ID', #exact path to the key inside the GeoJSON structure
    fill_color='viridis',  
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Biome Types South America'
).add_to(eco_region_base)


## add popup via an invisible GeoJSON layer
# source: https://python-visualization.github.io/folium/latest/user_guide/ui_elements/popups.html

folium.GeoJson(
    gdf_ecoregions_sa, 
    name = "Interactive Tooltips",
    # make polygons completely transparent, so they do not hide the choropleth colors
    style_function=lambda x:{"fillColor": "#ffffff00", "color":
"#ffffff00"},
tooltip = folium.GeoJsonTooltip(
    fields =["ECO_NAME", "BIOME_NAME"],
    aliases =['Region:', 'Biome:'],
    localize = True,
    ),
).add_to(eco_region_base)




## add layer control
folium.LayerControl().add_to(eco_region_base)


eco_region_base.save("ecoregions_sa.html")




In [11]:
## here popups instead of tooletips are used

# initialize basemap centerd on South America, add a zoom control limit
min_lon, max_lon = -81.5, -35.0
min_lat, max_lat = -65.0, 20.5


eco_region_base = folium.Map(
    max_bounds = True,
    location = [9, -64],  #7 = latitute, 7 degrees north of equator, -65 = longitude, 65 degrees west of the prime meridian
    zoom_start = 3, 
    tiles = "CartoDB Positron",
    control_scale = True,
    min_lat = min_lat,
    max_lat = max_lat,
    min_lon = min_lon,
    max_lon = max_lon
                         
)

## add choropleth map
folium.Choropleth(
    geo_data=gdf_ecoregions_sa,
    data = gdf_ecoregions_sa,
    name = "Biome Types South America",
    columns=['ECO_ID', 'BIOME_NUM'],
    key_on='feature.properties.ECO_ID', #exact path to the key inside the GeoJSON structure
    fill_color='viridis',  
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Biome Types South America'
).add_to(eco_region_base)


## add popup via an invisible GeoJSON layer
# source: https://python-visualization.github.io/folium/latest/user_guide/ui_elements/popups.html

folium.GeoJson(
    gdf_ecoregions_sa, 
    name = "Interactive Tooltips",
    # make polygons completely transparent, so they do not hide the choropleth colors
    style_function=lambda x:{"fillColor": "#ffffff00", "color":
"#ffffff00"},
popup = folium.GeoJsonPopup(
    fields =["ECO_NAME", "BIOME_NAME"],
    aliases =['Region:', 'Biome:'],
    localize = True,
    ),
).add_to(eco_region_base)




## add layer control
folium.LayerControl().add_to(eco_region_base)


eco_region_base.save("ecoregions_sa_popup.html")




#### **Combine Fires with Biome-Map**

1. Check the crs of both gdfs -> must should EPSG: 4326
2. Perform spatial join according to some conditions
3. Count how many fires are per biome
4. visualize it?

In [12]:
 ## check CRS of Fire GDF and Biome GDF
print(gdf_fire.crs)
print(gdf_ecoregions_sa.crs)


EPSG:4326
EPSG:4326


In [ ]:
## perform join according to conditions (See Lesson 7)

# keep Fire geometry and attach Biome geometry
# where the Fire is "within" the biome polygon

fires_within_biomes = gpd.sjoin(gdf_fire, gdf_ecoregions_sa,
                                how = "inner", predicate = "within")

fires_within_biomes.head(3)

,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,...,ECO_BIOME_,NNH,ECO_ID,SHAPE_LENG,SHAPE_AREA,NNH_NAME,COLOR,COLOR_BIO,COLOR_NNH,LICENSE
0,-5.44360,-36.79174,303.93,0.41,0.61,2026-05-14,313,N20,VIIRS,n,...,NO02,4,525,113.696455,60.155581,Nature Imperiled,#81B50A,#CCCD65,#EE1E23,CC-BY 4.0
1,-12.40172,-38.34732,302.24,0.40,0.60,2026-05-14,315,N20,VIIRS,n,...,NO01,3,442,47.691037,9.272023,Nature Could Recover,#2E5D00,#38A700,#F9A91B,CC-BY 4.0
2,-12.39697,-38.34255,309.97,0.40,0.60,2026-05-14,315,N20,VIIRS,n,...,NO01,3,442,47.691037,9.272023,Nature Could Recover,#2E5D00,#38A700,#F9A91B,CC-BY 4.0


In [18]:
## count the number of fires per biome

fires_per_biome = (
    fires_within_biomes.groupby("BIOME_NAME").size().reset_index(name="FIRE_COUNT")
)

fires_per_biome = fires_per_biome.sort_values("FIRE_COUNT", ascending=False)

display(fires_per_biome)

#.size = counts rows per group, .sum() summs up numerical values



,BIOME_NAME,FIRE_COUNT
8,"Tropical & Subtropical Grasslands, Savannas & ...",2425
9,Tropical & Subtropical Moist Broadleaf Forests,1515
7,Tropical & Subtropical Dry Broadleaf Forests,883
0,Deserts & Xeric Shrublands,863
1,Flooded Grasslands & Savannas,480
5,Temperate Broadleaf & Mixed Forests,348
6,"Temperate Grasslands, Savannas & Shrublands",274
3,"Mediterranean Forests, Woodlands & Scrub",170
4,Montane Grasslands & Shrublands,53
2,Mangroves,19


#### **Merge Fire Counts with Fires Within**

This merge is necessary so the fire counts can be included in the popup of the map

In [24]:
## merge fires_per_biomes with fires_within biomes

fires_within_biomes = fires_within_biomes.merge(
    fires_per_biome, 
    on = "BIOME_NAME",
    how = "left"
)

display(fires_within_biomes.head(3))

,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,...,ECO_ID,SHAPE_LENG,SHAPE_AREA,NNH_NAME,COLOR,COLOR_BIO,COLOR_NNH,LICENSE,FIRE_COUNT_x,FIRE_COUNT_y
0,-5.44360,-36.79174,303.93,0.41,0.61,2026-05-14,313,N20,VIIRS,n,...,525,113.696455,60.155581,Nature Imperiled,#81B50A,#CCCD65,#EE1E23,CC-BY 4.0,883,883
1,-12.40172,-38.34732,302.24,0.40,0.60,2026-05-14,315,N20,VIIRS,n,...,442,47.691037,9.272023,Nature Could Recover,#2E5D00,#38A700,#F9A91B,CC-BY 4.0,1515,1515
2,-12.39697,-38.34255,309.97,0.40,0.60,2026-05-14,315,N20,VIIRS,n,...,442,47.691037,9.272023,Nature Could Recover,#2E5D00,#38A700,#F9A91B,CC-BY 4.0,1515,1515


#### **Create Map With Fire Information Included**



In [26]:
## in pop-ups fire information is included


# initialize basemap centerd on South America, add a zoom control limit
min_lon, max_lon = -81.5, -35.0
min_lat, max_lat = -65.0, 20.5


eco_region_base2 = folium.Map(
    max_bounds = True,
    location = [9, -64],  #7 = latitute, 7 degrees north of equator, -65 = longitude, 65 degrees west of the prime meridian
    zoom_start = 3, 
    tiles = "CartoDB Positron",
    control_scale = True,
    min_lat = min_lat,
    max_lat = max_lat,
    min_lon = min_lon,
    max_lon = max_lon
                         
)

## add choropleth map
folium.Choropleth(
    geo_data=fires_within_biomes ,
    data = fires_within_biomes ,
    name = "Biome Types South America",
    columns=['ECO_ID', 'BIOME_NUM'],
    key_on='feature.properties.ECO_ID', #exact path to the key inside the GeoJSON structure
    fill_color='viridis',  
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Biome Types South America'
).add_to(eco_region_base2)


## add popup via an invisible GeoJSON layer
# source: https://python-visualization.github.io/folium/latest/user_guide/ui_elements/popups.html

folium.GeoJson(
    fires_within_biomes, 
    name = "Interactive Popups",
    # make polygons completely transparent, so they do not hide the choropleth colors
    style_function=lambda x:{"fillColor": "#ffffff00", "color":
"#ffffff00"},
popup = folium.GeoJsonPopup(
    fields =["ECO_NAME", "BIOME_NAME", "FIRE_COUNT_x" ],
    aliases =['Region:', 'Biome:', "Total #Fires per Biome:"],
    localize = True,
    ),
).add_to(eco_region_base2)


fires_per_biome



## add layer control
folium.LayerControl().add_to(eco_region_base2)


eco_region_base.save("ecoregions_sa_popup2.html")


